In [52]:
import pandas as pd
import os
import numpy as np
from configs.constants import MCBOOST_NAME

MCBOOST_TABLE_NAME = 'MCGrad'

In [53]:
# res_dir = 'mc_industry_results/tuned/'
res_dir = 'mc_industry_results/'
res_files = os.listdir(res_dir)

In [55]:
dfs = []
for f in res_files:
    if not f.endswith('.pkl'):
        continue
    dfs.append(pd.read_pickle(os.path.join(res_dir, f)))
df = pd.concat(dfs)
df = df[df.seed == 15].copy()
df['mcb_algorithm'] = df.mcb_algorithm.fillna('base_model')

In [56]:
# Categorize the different mcboost variants based on parameter values
def categorize_mcboost(mcb_algorithm, params):
    if mcb_algorithm == 'HKRR':
        return f'HKRR_{params["lambda"]}_{params["alpha"]}'
    if mcb_algorithm != MCBOOST_NAME:
        return mcb_algorithm
    if mcb_algorithm == MCBOOST_NAME:
        return params['name']

    print(params)
    return None

In [29]:
resolved_algos = [categorize_mcboost(row['mcb_algorithm'], row['mcb_algorithm_params']) for _, row in df.iterrows()]

In [30]:
df['mcb_algorithm'] = resolved_algos

In [31]:
df.mcb_algorithm.value_counts()

mcb_algorithm
base_model                       328
HKRR_0.1_0.1                     328
HKRR_0.1_0.05                    328
HKRR_0.1_0.025                   328
HKRR_0.1_0.0125                  328
Isotonic                         328
CASMCBoost_msh_0_ablation        328
CASMCBoost                       328
CASMCBoost_group_features        328
CASMCBoost_unshrink_ablation     328
CASMCBoost_one_round_ablation    328
DFMCBoost                        328
Name: count, dtype: int64

In [32]:
df = df[df.mcb_algorithm.notnull()]

In [33]:
df_test = df[df.set_name == 'test']
### For HKRR take the validation results to find best params per dataset and remove other results from the test results
df_val_hkrr = df[(df.set_name == 'validation') & (df.mcb_algorithm.str.startswith('HKRR')) & (df.group == 'agg')].reset_index()
best_hkrr_params_per_dataset = df_val_hkrr.iloc[df_val_hkrr.groupby('dataset')['logloss'].idxmin()][['dataset', 'mcb_algorithm']].assign(matched=True)
df_test = pd.merge(df_test, best_hkrr_params_per_dataset, on=['dataset', 'mcb_algorithm'], how='left')
df_test['select'] = df_test.apply(lambda row: not np.isnan(row.matched) or not row.mcb_algorithm.startswith('HKRR'), axis=1)
df_test = df_test[df_test.select]
df_test = df_test.drop(columns=['matched', 'select'])
df_test['mcb_algorithm'] = df_test.mcb_algorithm.apply(lambda x: x if not x.startswith('HKRR') else 'HKRR')

In [34]:
# Check if we're left with one result per dataset per algorithm on aggregated test data
(df_test[df_test.group == 'agg'].groupby(['dataset', 'mcb_algorithm']).group.count() != 1).any()

np.False_

In [35]:
res_tab_index_cols = ['dataset', 'model', 'mcb_algorithm']
metric_name_mapping = {
    'MCE_perc': 'MCE_perc_features',
    'MCE_sigma': 'MCE_sigma_features',
    'ECCE_perc': 'global_ECCE_perc',
    'ECCE_sigma': 'global_ECCE_sigma',
    'ECE': 'global_ECE',
    'logloss': 'logloss',
    'prauc': 'prauc',
    'fit_time': 'fit_time',
    'num_rounds': 'num_rounds_mcboost',

}
agg_df = df_test[df_test.group == 'agg']

res_tab = agg_df[res_tab_index_cols + list(metric_name_mapping.keys()) + ['mcb_algorithm_params']].set_index(res_tab_index_cols)
res_tab = res_tab.rename(columns=metric_name_mapping)

max_df = df_test[df_test.group == 'max']
max_df_sel = max_df[res_tab_index_cols + ['ECCE_perc', 'ECCE_sigma', 'smECE']].set_index(res_tab_index_cols)
max_df_sel = max_df_sel.rename(columns={'ECCE_perc': 'MCE_perc_groups', 'ECCE_sigma': 'MCE_sigma_groups', 'smECE': 'max_group_smECE'})
res_tab = res_tab.join(max_df_sel, how='left')

In [36]:
# Check there are no extra rows
(res_tab.reset_index(drop=False).groupby(['dataset', 'mcb_algorithm']).prauc.count() != 1).any()

np.False_

In [37]:
# Take just LogisticRegression
lr_res_tab = res_tab.xs('LogisticRegression', level=1)

In [38]:
pretty_colnames = {
    'global_ECCE_perc': 'ECCE-%',
    'global_ECCE_sigma': 'ECCE-σ',
    'global_ECE': 'ECE',
    'logloss': 'Log-Loss',
    'prauc': 'PRAUC',
    'MCE_perc_features': 'MCE-% (all)',
    'MCE_sigma_features': 'MCE-σ (all)' ,
    'MCE_perc_groups': 'MCE-% (grp)',
    'MCE_sigma_groups': 'MCE-σ (grp)',
    'max_group_smECE': 'max smECE (grp)',
}

In [51]:
MCBOOST_NAME

'CASMCBoost'

In [76]:
# temporary rename
# lr_res_tab.reset_index(drop=False, inplace=True)
# lr_res_tab = lr_res_tab.rename({'CASMCBoost_msh_0_ablation': 'CASMCBoost_msh_20', 'CASMCBoost_unshrink_ablation': 'CASMCBoost_no_unshrink', 'CAS_MCBoost_one_round_ablation': 'CASMCBoost_one_round'})
# lr_res_tab.set_index(['dataset', 'mcb_algorithm'], inplace=True)

In [83]:
lr_res_tab.to_csv('kdd_mcboost_results_fixed.csv', index=True)

In [78]:
global_metrics_res_tab = lr_res_tab[['global_ECCE_perc', 'global_ECCE_sigma', 'global_ECE', 'logloss', 'prauc']].rename(columns=pretty_colnames)
mc_metrics_res_tab = lr_res_tab[['MCE_perc_features', 'MCE_sigma_features', 'MCE_perc_groups', 'MCE_sigma_groups', 'max_group_smECE']].rename(columns=pretty_colnames)

### Latex tables

In [79]:
import pandas as pd
import re

def escape_latex(text):
    """Escape LaTeX special characters in text."""
    if not isinstance(text, str):
        return text
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }
    pattern = re.compile('|'.join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group()], text)

def multiindex_to_latex_highlighted(df, digits=4):
    """Convert MultiIndex DataFrame to LaTeX tabular format with per-group highlights and lines after each top-level group."""
    higher_is_better = ['PRAUC']
    numeric_cols = df.select_dtypes(include='number').columns

    group_level = 0
    group_min, group_max = {}, {}

    for name, group in df.groupby(level=group_level):
        for col in numeric_cols:
            if col in higher_is_better:
                group_max[(name, col)] = group[col].max()
            else:
                group_min[(name, col)] = group[col].min()

    latex_lines = []
    last_values = [""] * df.index.nlevels

    headers = [escape_latex(h) for h in list(df.index.names) + list(df.columns)]
    latex_lines.append("\\begin{tabular}{" + "l" * len(headers) + "}")
    latex_lines.append("\\toprule")
    latex_lines.append(" & ".join(headers) + " \\\\")
    latex_lines.append("\\midrule")

    current_group = None
    for idx, row in df.iterrows():
        top_group = idx[group_level]

        # If a new group starts, insert a line before it (but not before the first group)
        if current_group is not None and top_group != current_group:
            latex_lines.append("\\midrule")
        current_group = top_group

        row_items = []
        for level, value in enumerate(idx):
            if value == last_values[level]:
                row_items.append("")
            else:
                row_items.append(escape_latex(str(value)))
                last_values[level] = value
            for reset_level in range(level + 1, df.index.nlevels):
                last_values[reset_level] = ""

        for col in df.columns:
            val = row[col]
            if isinstance(val, float):
                formatted = f"{val:.{digits}f}"
                if col in higher_is_better and val == group_max.get((top_group, col), None):
                    formatted = f"\\textbf{{{formatted}}}"
                elif col not in higher_is_better and val == group_min.get((top_group, col), None):
                    formatted = f"\\textbf{{{formatted}}}"
            else:
                formatted = escape_latex(str(val))
            row_items.append(formatted)

        latex_lines.append(" & ".join(row_items) + " \\\\")

    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    return "\n".join(latex_lines)


In [80]:
print(multiindex_to_latex_highlighted(mc_metrics_res_tab))

\begin{tabular}{lllllll}
\toprule
dataset & mcb\_algorithm & MCE-\% (all) & MCE-σ (all) & MCE-\% (grp) & MCE-σ (grp) & max smECE (grp) \\
\midrule
MEPS & base\_model & 15.2178 & 3.9940 & 45.7745 & 3.2849 & 0.0819 \\
 & HKRR & 18.2990 & 4.8624 & \textbf{32.4973} & \textbf{3.1814} & 0.1054 \\
 & Isotonic & \textbf{13.6096} & \textbf{3.6076} & 43.6074 & 3.4070 & \textbf{0.0713} \\
 & CASMCBoost\_msh\_20 & 14.5510 & 3.8815 & 37.5404 & 3.2923 & 0.0755 \\
 & CASMCBoost & 15.2178 & 3.9940 & 45.7745 & 3.2849 & 0.0819 \\
 & CASMCBoost\_group\_features & 15.2178 & 3.9940 & 45.7745 & 3.2849 & 0.0819 \\
 & CASMCBoost\_no\_unshrink & 15.2178 & 3.9940 & 45.7745 & 3.2849 & 0.0819 \\
 & CASMCBoost\_one\_round\_ablation & 14.7317 & 3.9250 & 38.9465 & 3.2888 & 0.0733 \\
 & DFMCBoost & 13.7272 & 3.6171 & 46.6125 & 3.3323 & 0.0743 \\
\midrule
acs\_employment\_all\_states & base\_model & 6.0280 & 87.8687 & 34.9800 & 48.9901 & 0.0505 \\
 & HKRR & 4.5304 & 67.4985 & 29.7766 & 16.7063 & 0.0129 \\
 & Isotonic 

In [74]:
print(multiindex_to_latex_highlighted(global_metrics_res_tab))

\begin{tabular}{lllllll}
\toprule
dataset & mcb\_algorithm & ECCE-\% & ECCE-σ & ECE & Log-Loss & PRAUC \\
\midrule
MEPS & base\_model & 9.0583 & 2.3774 & 0.0215 & 0.3177 & 0.6211 \\
 & HKRR & 9.5242 & 2.5308 & 0.0252 & 0.3367 & 0.5317 \\
 & Isotonic & \textbf{6.4659} & \textbf{1.7140} & \textbf{0.0118} & 0.3160 & 0.5945 \\
 & CASMCBoost\_msh\_20 & 8.4270 & 2.2479 & 0.0186 & \textbf{0.3150} & \textbf{0.6221} \\
 & CASMCBoost & 9.0583 & 2.3774 & 0.0215 & 0.3177 & 0.6211 \\
 & CASMCBoost\_group\_features & 9.0583 & 2.3774 & 0.0215 & 0.3177 & 0.6211 \\
 & CASMCBoost\_no\_unshrink & 9.0583 & 2.3774 & 0.0215 & 0.3177 & 0.6211 \\
 & CASMCBoost\_one\_round\_ablation & 8.0368 & 2.1413 & 0.0189 & 0.3153 & 0.6204 \\
 & DFMCBoost & 7.4256 & 1.9566 & 0.0201 & 0.3155 & 0.6190 \\
\midrule
acs\_employment\_all\_states & base\_model & 3.6483 & 53.1801 & 0.0332 & 0.2258 & 0.9274 \\
 & HKRR & 0.5298 & 7.8937 & 0.0057 & 0.2105 & 0.9234 \\
 & Isotonic & 0.2180 & 3.3114 & \textbf{0.0010} & 0.2059 & 0.9295 \